In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")


### State Backend

In [2]:
import os
from deepagents import create_deep_agent
from deepagents.backends import StateBackend


In [4]:
## 1. Create the agent -  these two are equivalent

agent = create_deep_agent(model="groq:meta-llama/llama-4-scout-17b-16e-instruct")

## Under the hood this is what 'agent' is doing - explicit StateBackend:

agent2 = create_deep_agent(
    model="groq:meta-llama/llama-4-scout-17b-16e-instruct",
    backend=StateBackend(),
)



In [5]:
## 2. Invoke the agent and ask it to WRITE a file
## StateBeckendkeeps that files inside LangGraph State

result = agent2.invoke({"messages":[{
    "role":"user",
    "content": (
        "Create a file at /notes/todo.txt with the exactly this content:\n"
        "1. Record Video\n2. Edit Video\n3. Upload Video\n"
        "Then tell me when you've done it."
    )
}]})

## The agent's final natural-language reply
print("\n --Agent's Reply--")
print(result["messages"][-1].content)


 --Agent's Reply--
I've created the file /notes/todo.txt with the specified content:
1. Record Video
2. Edit Video
3. Upload Video


In [7]:
## 3. Check the backend is working
## With StateBackend, written files appear inder results["files"]

print("\n--Backend Check--")
files = result.get("files", {})

if files:
    print(f"✅ StateBackend is working - {len(files)} file(s) is State:")
    for path, content in files.items():
        print(f"\n📄 {path}\n{'-'*40}\n{content}")
else:
    print("⚠️ No files found in State. Either the agent didn't write the file(s),"
    "or the backend isn't wired up correctly.")


--Backend Check--
✅ StateBackend is working - 1 file(s) is State:

📄 /notes/todo.txt
----------------------------------------
{'content': '1. Record Video\n2. Edit Video\n3. Upload Video', 'encoding': 'utf-8', 'created_at': '2026-06-28T10:27:27.275494+00:00', 'modified_at': '2026-06-28T10:27:27.275494+00:00'}


In [9]:
## 4. Prove Persistence WITHIN the same thread
## feed the returned State back in and ask it to READ the files

followup = agent2.invoke({
    "messages":[{
        "role":"user",
        "content": "Read /notes/todo.txt back to me."
    }],
    "files": result.get("files", {}),  # <-- Pass the virtual filesystem along
})

print("\n--READ BACK (Same Thread)--")
print(followup["messages"][-1].content)


--READ BACK (Same Thread)--
1. Record Video
2. Edit Video
3. Upload Video


### FileSystemBackend(local disk)

In [12]:
## 1. Create the agent with real-disk backend
## root_dir="." -> Files land relative to your current working directory
## virtual_mode=True -> agent uses virtual paths likes /notes/todo.txt,
##                      mapped onto 

from deepagents.backends import FilesystemBackend
ROOT="."

agent = create_deep_agent(
    model="groq:meta-llama/llama-4-scout-17b-16e-instruct",
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True)
)
print(f"✅ Agent created with FileSysytembackend(root_dir={ROOT!r}).")
print("   Files wirtten by the agent will appear on your ACTUAL disk.")

✅ Agent created with FileSysytembackend(root_dir='.').
   Files wirtten by the agent will appear on your ACTUAL disk.


In [13]:
## 2. Invoke the agent and ask it to WRITE a file

result = agent.invoke({"messages":[{
    "role":"user",
    "content": (
        "Create a file at /notes/todo.txt with the exactly this content:\n"
        "1. Record Video\n2. Edit Video\n3. Upload Video\n"
        "Then tell me when you've done it."
    )
}]})

## The agent's final natural-language reply
print("\n --Agent's Reply--")
print(result["messages"][-1].content)


 --Agent's Reply--
I've created the file /notes/todo.txt with the specified content:
1. Record Video
2. Edit Video
3. Upload Video


In [15]:
## 3. Check the backend is working - look on the REAL disk
## With the virtual_mode=True, /notes/todo.txt maps to ./notes/todo.txt

from pathlib import Path
print("\n--Backend Check (Real Filesystem)--")
disk_path = Path(ROOT) / "notes" / "todo.txt"

if disk_path.exists():
    print(f"✅ FileSystemBackend is working - File exist on disk")
    print(f"📄 {disk_path.resolve()}\n{'-'*40}")
    print(disk_path.read_text())
else:
    print(f"⚠️ Expected file not found at {disk_path.resolve()}")
    print("   The agent may not have called the write tool, or the path"
    "mapping diifers.")



--Backend Check (Real Filesystem)--
✅ FileSystemBackend is working - File exist on disk
📄 /Users/nishantsingh04/Documents/Deep Agent/deepagentsdemo/notes/todo.txt
----------------------------------------
1. Record Video
2. Edit Video
3. Upload Video


In [16]:
## 4. Prove Persistence WITHIN the same thread
## Unlike StateBackend, this file survives even after Python exits.
## A brand-new agent (fresh-state) can read it back from the disk

fresh_agent = create_deep_agent(
    model="groq:meta-llama/llama-4-scout-17b-16e-instruct",
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True)
)

followup = fresh_agent.invoke({
    "messages":[{
        "role":"user",
        "content": "Read /notes/todo.txt back to me."
    }],
    # NOTE: no 'files' state is passed in - the file is read straight from disk
})

print("\n--READ BACK with FRESH agent (proves disk persistence)--")
print(followup["messages"][-1].content)


--READ BACK with FRESH agent (proves disk persistence)--
1. Record Video
2. Edit Video
3. Upload Video


### StoreBackend

Creates a deep agent backend by a LangGraph store, invokes it to erite a file on a one thread, then proves the backend works by reading that file bakc on a DIFFERENT thread -- something StateBackend cannot do.


In [17]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend
store = InMemoryStore()

agent = create_deep_agent(
    model="groq:meta-llama/llama-4-scout-17b-16e-instruct",
    backend=StoreBackend(
        ## Local dev: static namespace. No developmwnt runtime needed.
        ## In a LangSmith Development you'd use the user-identity version.
        namespace=lambda rt: ("demo-user",),
    ),
    store=store,
)

print("✅ Agent created with StoreBackend + Static namespace.")

✅ Agent created with StoreBackend + Static namespace.


In [18]:
import os
import uuid

## 2. THREAD 1 - Write a file
thread_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = agent.invoke(
    {
        "messages": [{
            "role":"user",
            "content": (
                "Create a file at /notes/todo.txt with the exactly this content:\n"
                "1. Record Video\n2. Edit Video\n3. Upload Video\n"
                "Then tell me when you've done it."
            )
        }]
    }, config= thread_1
)

print("\n--Agent reply (thread 1)--")
print(result["messages"][-1].content)


--Agent reply (thread 1)--
I've created the file /notes/todo.txt with the specified content:

1. Record Video
2. Edit Video
3. Upload Video


In [19]:
## Thread 2: Read back on a different thread

thread_2 = {"configurable": {"thread_id":str(uuid.uuid4())}}

followup = agent.invoke({
    "messages":[{
        "role":"user",
        "content": "Read /notes/todo.txt back to me."
    }]
   
}, config=thread_2
)

print("\n--READ BACK on a different thread--")
print(followup["messages"][-1].content)


--READ BACK on a different thread--
1. Record Video
2. Edit Video
3. Upload Video


With StoreBackend + InMemoryStore, the file is not saved to disk at all. It lives in rAM, inside the InMemoryStore object, as an entry keyed under the namespace

# Deep Agent Backends — Where Your Files Actually Live

Every Deep Agent works with a **virtual filesystem**: its tools read and write
files using paths like `/notes/todo.txt`. But those paths are an abstraction.
The **backend** decides where that data physically lives — and that single
choice changes everything about persistence, sharing, and durability.

The agent code stays identical across all three. Only the backend changes.

---

## 1. StateBackend (default)

Files live **in the agent's LangGraph state** — i.e. in RAM, tied to a single
thread. They're available during the run via `result["files"]`, and you must
manually carry that state forward to keep using them.

- **Where:** in-memory dict, inside one thread's state
- **Survives:** within a single conversation/thread only
- **Gone when:** the thread ends or the process exits
- **Use it for:** ephemeral scratch space, temporary working files

## 2. FilesystemBackend

Files are written to your **real disk**, under `root_dir`. With
`virtual_mode=True`, a virtual path like `/notes/todo.txt` maps to
`root_dir/notes/todo.txt`. These are genuine files you can open in an editor
or `cat` from a terminal.

- **Where:** your actual filesystem, relative to `root_dir`
- **Survives:** across process restarts — it's a real file
- **Use it for:** when the agent should edit real project files
- **⚠️ Caution:** grants the agent real read/write disk access — point it at a
  scratch directory, not anything important

## 3. StoreBackend

Files live in a **LangGraph store**, scoped by a `namespace`. This is the only
backend whose files persist **across threads/conversations** — write on one
thread, read back on another. With `InMemoryStore` this works within a single
process; swap in a persistent store (e.g. Postgres-backed) for true
cross-restart durability.

- **Where:** inside the store object, under a namespace key
- **Survives:** across threads sharing the same store
- **Gone when:** the process exits (if using `InMemoryStore`) — use a
  persistent store to outlive restarts
- **Use it for:** long-term memory, per-user file spaces

---

### Quick comparison

| Backend            | Lives in        | Cross-thread? | Survives restart?      | Real file on disk? |
|--------------------|-----------------|:-------------:|:----------------------:|:------------------:|
| `StateBackend`     | LangGraph state |       ❌      |           ❌           |         ❌         |
| `FilesystemBackend`| Your disk       |       ✅      |           ✅           |         ✅         |
| `StoreBackend`     | A LangGraph store |     ✅      | only w/ persistent store |       ❌         |

> **Mental model:** the agent never knows the difference. Its tools always
> speak in file paths — the backend quietly translates every read/write into
> the right operation underneath. That's what lets you swap backends without
> touching a line of agent logic.